# Hate Speech Classification

Done by: Muhammad Shariff Bin Abdul Rashid
---
Student ID: 01443150
---
SMU IS469 Section G2
---
Gen AI with LLMs Individual Assignment
---

This notebook explores different approaches to classifying hate speech, specifically focusing on fine-tuning techniques like full fine-tuning and LoRA (Low-Rank Adaptation). We also evaluate a model pre-trained specifically on a hate speech dataset for comparison.

## Project Overview

The goal is to build and evaluate models capable of classifying text into one of three categories: "hatespeech", "normal", or "offensive", based on the [HateXplain dataset](https://huggingface.co/datasets/Hate-speech-CNERG/hatexplain).

## Model Choice and Design Decisions

For this text classification task, we primarily utilize **encoder-only transformer models** like RoBERTa and BERT. Encoder models are well-suited for tasks that require understanding the entire input sequence to make a prediction, such as classification, sentiment analysis, or named entity recognition. Unlike decoder-only models (which are good for text generation) or encoder-decoder models (used for sequence-to-sequence tasks like translation), encoder models process the input text in a non-autoregressive manner, building a rich representation of the input which is then fed into a classification head. This makes them efficient and effective for assigning a single label to an input text.

Other key decisions made in this notebook include:

*   **Using pre-trained models:** Leveraging models pre-trained on large text corpora (like RoBERTa-base and BERT-base) provides a strong starting point and significantly reduces the amount of data and computation needed for fine-tuning.
*   **Fine-tuning approaches:** Exploring both **full fine-tuning** (updating all model parameters) and **LoRA fine-tuning** (updating a small subset of parameters) to compare their performance and efficiency trade-offs.
*   **Weighted Cross-Entropy Loss:** Addressing the class imbalance in the HateXplain dataset by using weighted cross-entropy loss during training, which gives more importance to the less frequent classes.
*   **Evaluation Metrics:** Using a combination of **Accuracy** and **Macro F1-score** for evaluation. Macro F1 is particularly important for imbalanced datasets as it provides a more reliable measure of performance across all classes compared to micro F1 or accuracy alone.

## Data Loading and Preprocessing

The notebook starts by loading the HateXplain dataset. A custom normalization function is applied to process the raw data, extract the text, majority-voted label, rationale mask, and targets. Empty text entries are filtered out.

## Model Architectures and Fine-tuning Strategies

We experiment with the following models and fine-tuning strategies:

1.  **Full Fine-tuning:** The entire pre-trained transformer model's parameters are updated during training on the HateXplain dataset. We use a RoBERTa-base model for this.
2.  **LoRA Fine-tuning:** Only a small number of additional parameters are trained by injecting low-rank matrices into specific layers of the pre-trained RoBERTa-base model. This significantly reduces the number of trainable parameters and can be more memory-efficient.
3.  **Pre-trained HateXplain Model:** We evaluate a model (`Hate-speech-CNERG/bert-base-uncased-hatexplain`) that has already been fine-tuned on the HateXplain dataset. This serves as a benchmark for comparison.

## Key Learning Points and Considerations

*   **Dataset Nuances:** The HateXplain dataset provides valuable rationale masks and targets, which can be used for explainability or potentially incorporated into the training process (though not done in the main classification evaluation here).
*   **Class Imbalance:** The dataset has some class imbalance (as seen in the class counts), which is addressed by using weighted cross-entropy loss during training.
*   **Fine-tuning Approaches:**
    *   **Full Fine-tuning:** Generally yields good performance but requires updating a large number of parameters, making it computationally expensive and memory-intensive.
    *   **LoRA Fine-tuning:** Offers a more efficient alternative by training only a small fraction of parameters. This can be beneficial for memory-constrained environments or when experimenting with multiple tasks. In this notebook, LoRA achieved slightly lower performance compared to full fine-tuning but with significantly fewer trainable parameters.
    *   **Pre-trained Task-Specific Models:** Using models already fine-tuned on similar tasks (like the HateXplain model) can provide a strong baseline and sometimes even outperform models fine-tuned from general-purpose pre-trained checkpoints, depending on the task similarity and the quality of the pre-training.
*   **GPU Memory Management:** Training and evaluating large transformer models can quickly consume GPU memory. Techniques like reducing batch size, explicitly deleting variables, clearing the CUDA cache (`torch.cuda.empty_cache()`), and using manual evaluation loops (as demonstrated to overcome memory errors) are crucial.
*   **Evaluation Metrics:** Accuracy provides an overall measure of correct predictions, while Macro F1 is particularly useful for imbalanced datasets as it calculates the F1 score for each class and then averages them, giving equal weight to each class regardless of its size. Precision and Recall provide further insight into the types of errors the model makes.

## Differences Between LoRA and Full Fine-tuning in this Case

*   **Trainable Parameters:** LoRA trains a much smaller number of parameters compared to full fine-tuning. This is evident in the `print_trainable_parameters()` output for the LoRA model.
*   **Performance:** In this specific experiment, full fine-tuning with RoBERTa achieved a slightly higher macro F1 score and accuracy on the validation set compared to LoRA fine-tuning with the same base model. This is not always the case, and LoRA can sometimes match or even exceed the performance of full fine-tuning, especially with careful hyperparameter tuning of the LoRA configuration.
*   **Computational Cost:** LoRA training is generally faster and requires less memory than full fine-tuning due to the reduced number of parameters being updated.



In [ ]:
!pip -q install "torch"
!pip -q install "datasets<3.0" "pyarrow<20" "transformers>=4.44,<4.46" "accelerate==0.34.2" "evaluate"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:

# @title Imports & checking versions
import torch, transformers, datasets, evaluate
import numpy as np, random, os
import accelerate
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from accelerate.utils.memory import clear_device_cache
from datasets import load_dataset, DatasetDict

# Reproducibility
SEED = 42
def set_seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed_all()
clear_device_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Device:", device)


Torch: 2.8.0+cu126
Transformers: 4.45.2
Accelerate: 0.34.2
Datasets: 2.21.0
Evaluate: 0.4.6
Device: cuda


### Data Loading and Preprocessing

In [ ]:
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)
print(ds)  # expect train/validation/test

### Normalization Function

In [ ]:
# @title Loading Data
import json

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    """
    Returns a list of dicts like: [{"label": int, "target": [...]}, ...]
    Handles:
      - list of dicts
      - dict of lists: {"label":[...], "target":[...], ...}
      - list of ints (labels)
      - JSON string of any of the above
      - anything else -> []
    """
    if ann is None:
        return []
    if isinstance(ann, str):
        try:
            ann = json.loads(ann)
        except Exception:
            return []
    if isinstance(ann, list):
        if len(ann) == 0:
            return []
        if isinstance(ann[0], dict):
            return ann
        # list of ints/strings
        out = []
        for x in ann:
            try:
                out.append({"label": int(x), "target": []})
            except Exception:
                pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        # normalize to same length
        n = len(labels)
        out = []
        for i in range(n):
            try:
                lab = int(labels[i])
            except Exception:
                continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    """
    Returns list[list[int]]; handles JSON string or empty.
    """
    if rats is None:
        return []
    if isinstance(rats, str):
        try:
            rats = json.loads(rats)
        except Exception:
            return []
    if isinstance(rats, list):
        # ensure nested lists of ints
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    # tie-breaker priority: hatespeech (0) > offensive (2) > normal (1)
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    # if nothing valid, default to normal (1)
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats:
        return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    # Prefer post_tokens if present; else fallbacks
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex:
        return str(ex["text"])
    if "cleaned_text" in ex:
        return str(ex["cleaned_text"])
    # fallback: join first list-of-strings
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,  # optional; safe to ignore later
        "targets": targets,          # optional
    }

# Map without trying to pre-remove columns (safer); we’ll trim afterwards
ds = ds.map(normalize_example, num_proc=2)

# Keep only what we need
to_keep = ["text_norm", "label_norm", "rationale_mask", "targets"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Drop empties
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

print(ds)

### Display Examples

In [ ]:
# @title
ds = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)
print(ds)  # expect train/validation/test


Generating train split:   0%|          | 0/15383 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1922 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1924 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1924
    })
})


In [ ]:
# @title
import json

ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v:k for k,v in ID2NAME.items()}

def _coerce_annotators(ann):
    """
    Returns a list of dicts like: [{"label": int, "target": [...]}, ...]
    Handles:
      - list of dicts
      - dict of lists: {"label":[...], "target":[...], ...}
      - list of ints (labels)
      - JSON string of any of the above
      - anything else -> []
    """
    if ann is None:
        return []
    if isinstance(ann, str):
        try:
            ann = json.loads(ann)
        except Exception:
            return []
    if isinstance(ann, list):
        if len(ann) == 0:
            return []
        if isinstance(ann[0], dict):
            return ann
        # list of ints/strings
        out = []
        for x in ann:
            try:
                out.append({"label": int(x), "target": []})
            except Exception:
                pass
        return out
    if isinstance(ann, dict):
        labels = ann.get("label", [])
        targets = ann.get("target", [])
        # normalize to same length
        n = len(labels)
        out = []
        for i in range(n):
            try:
                lab = int(labels[i])
            except Exception:
                continue
            tgt = targets[i] if (isinstance(targets, list) and i < len(targets)) else []
            out.append({"label": lab, "target": tgt if isinstance(tgt, list) else [tgt]})
        return out
    return []

def _coerce_rationales(rats):
    """
    Returns list[list[int]]; handles JSON string or empty.
    """
    if rats is None:
        return []
    if isinstance(rats, str):
        try:
            rats = json.loads(rats)
        except Exception:
            return []
    if isinstance(rats, list):
        # ensure nested lists of ints
        out = []
        for r in rats:
            if isinstance(r, list):
                out.append([int(v) if isinstance(v, (int, bool)) else 0 for v in r])
        return out
    return []

def _collect_targets(anns):
    seen, out = set(), []
    for a in anns:
        for t in a.get("target", []) or []:
            if t not in seen:
                seen.add(t); out.append(t)
    return out

def _majority_vote_label(anns):
    # tie-breaker priority: hatespeech (0) > offensive (2) > normal (1)
    pri = [0, 2, 1]
    counts = {0:0, 1:0, 2:0}
    for a in anns:
        lab = int(a.get("label", 1))
        if lab in counts: counts[lab] += 1
    # if nothing valid, default to normal (1)
    if counts[0] == counts[1] == counts[2] == 0:
        return 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], pri.index(kv[0])))[0][0]

def _majority_rationale(rats, n_tokens):
    if not rats:
        return [0] * n_tokens
    thresh = (len(rats) + 1) // 2
    agg = [0] * n_tokens
    for r in rats:
        for i, v in enumerate(r[:n_tokens]):
            agg[i] += int(v)
    return [1 if c >= thresh else 0 for c in agg]

def _pick_text(ex):
    # Prefer post_tokens if present; else fallbacks
    if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
        return " ".join(ex["post_tokens"])
    if "text" in ex:
        return str(ex["text"])
    if "cleaned_text" in ex:
        return str(ex["cleaned_text"])
    # fallback: join first list-of-strings
    for v in ex.values():
        if isinstance(v, list) and all(isinstance(t, str) for t in v):
            return " ".join(v)
    return ""

def normalize_example(e):
    tokens = e.get("post_tokens", None)
    text = _pick_text(e)
    anns = _coerce_annotators(e.get("annotators", None))
    rats = _coerce_rationales(e.get("rationales", None))
    label = _majority_vote_label(anns)
    rat_mask = _majority_rationale(rats, len(tokens) if isinstance(tokens, list) else len(text.split()))
    targets = _collect_targets(anns)
    return {
        "text_norm": text.strip(),
        "label_norm": int(label),
        "rationale_mask": rat_mask,  # optional; safe to ignore later
        "targets": targets,          # optional
    }

# Map without trying to pre-remove columns (safer); we’ll trim afterwards
ds = ds.map(normalize_example, num_proc=2)

# Keep only what we need
to_keep = ["text_norm", "label_norm", "rationale_mask", "targets"]
drop_cols = [c for c in ds["train"].column_names if c not in to_keep]
ds = ds.remove_columns(drop_cols)

# Drop empties
ds = ds.filter(lambda t: len(t.strip()) > 0, input_columns=["text_norm"])

print(ds)


Map (num_proc=2):   0%|          | 0/15383 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1922 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1924 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15383 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1922 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1924 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 1924
    })
})


In [ ]:
print(ds["train"][0])
print(ds["train"][12])
print(ds["train"][42])

{'text_norm': 'u really think i would not have been raped by feral hindu or muslim back in india or bangladesh and a neo nazi would rape me as well just to see me cry', 'label_norm': 2, 'rationale_mask': [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], 'targets': ['Hindu', 'Islam', 'Other']}
{'text_norm': 'you are completely correct without it communication breaks down leading to distortions misrepresentations this is a very serious problem among white males who failed to learn the language effectively', 'label_norm': 1, 'rationale_mask': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'targets': ['None', 'Caucasian']}
{'text_norm': '<user> you need potassium eat a banana you fucking nigger', 'label_norm': 0, 'rationale_mask': [0, 0, 0, 0, 0, 0, 0, 1, 1, 1], 'targets': ['African']}


In [ ]:
# @title
from IPython.display import display, HTML

def pretty_print_example(example):
    text = example['text_norm']
    mask = example['rationale_mask']
    label = ID2NAME[example['label_norm']]
    targets = example['targets']

    # Simple tokenization to align with the mask
    tokens = text.split()

    highlighted_text = []
    for i, token in enumerate(tokens):
        if i < len(mask) and mask[i] == 1:
            highlighted_text.append(f'<mark>{token}</mark>')
        else:
            highlighted_text.append(token)

    formatted_text = ' '.join(highlighted_text)

    html_output = f"""
    <p><strong>Text:</strong> {formatted_text}</p>
    <p><strong>Label:</strong> {label}</p>
    <p><strong>Targets:</strong> {', '.join(targets)}</p>
    """
    display(HTML(html_output))

# Now you can use this function to display your examples
print("Example 0:")
pretty_print_example(ds["train"][0])

print("\nExample 12:")
pretty_print_example(ds["train"][12])

print("\nExample 42:")
pretty_print_example(ds["train"][42])

Example 0:



Example 12:



Example 42:


In [ ]:
MODEL_NAME = "bert-base-uncased"
num_labels = len(ID2NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)
model.to(device);


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import torch.nn as nn

if model.num_labels != num_labels:
    in_features = model.classifier.in_features  # BERT's pooled output size
    model.classifier = nn.Linear(in_features, num_labels)
    model.config.num_labels = num_labels

# Nice label maps for logs
model.config.id2label = ID2NAME
model.config.label2id = NAME2ID
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")
prec= evaluate.load("precision")
rec = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "macro_precision": prec.compute(predictions=preds, references=labels, average="macro")["precision"],
        "macro_recall": rec.compute(predictions=preds, references=labels, average="macro")["recall"],
    }
    # per-class F1
    labels_np = np.array(labels)
    # Use the values from ID2NAME as label names
    for i, name in enumerate(ID2NAME.values()):
        out[f"f1_{name}"] = f1.compute(
            predictions=(preds==i).astype(int),
            references=(labels_np==i).astype(int),
            average="binary"
        )["f1"]
    return out

In [ ]:
USE_WEIGHTS = True
if USE_WEIGHTS:
    counts = np.bincount(np.array(ds["train"]["label_norm"]))
    class_weights = (1.0 / np.clip(counts, 1, None))
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    print("Train class counts:", counts, "→ weights:", class_weights)
else:
    class_weights = None

import torch.nn as nn
from transformers import Trainer

class WeightedCELoss(nn.Module):
    def __init__(self, weights=None):
        super().__init__()
        if weights is not None:
            # Move weights to the appropriate device
            self.weights = torch.tensor(weights, dtype=torch.float).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
            self.loss = nn.CrossEntropyLoss(weight=self.weights)
        else:
            self.loss = nn.CrossEntropyLoss()
    def forward(self, logits, labels):
        return self.loss(logits, labels)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.criterion = WeightedCELoss(class_weights) if class_weights is not None else nn.CrossEntropyLoss()
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = self.criterion(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

Train class counts: [4748 6251 4384] → weights: [1.05537642 0.8016201  1.14300348]


In [ ]:
MAX_LEN = 128

def tok_128(batch):
    enc = tokenizer(batch["text_norm"], truncation=True, padding=False, max_length=MAX_LEN)
    enc["labels"] = batch["label_norm"]
    return enc

tok_ds_128 = ds.map(tok_128, batched=True, remove_columns=ds["train"].column_names)
collator = DataCollatorWithPadding(tokenizer)
ds


Map:   0%|          | 0/15383 [00:00<?, ? examples/s]

Map:   0%|          | 0/1922 [00:00<?, ? examples/s]

Map:   0%|          | 0/1924 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['text_norm', 'label_norm', 'rationale_mask', 'targets'],
        num_rows: 1924
    })
})

In [ ]:
# @title Running the 4 different models
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd, numpy as np, torch, gc, time

candidates = [
    "roberta-base",
    # toxicity domain-adapted BERT
    "GroNLP/hateBERT",
    # efficient baseline
    "google/electra-base-discriminator",
    # your current reference
    "bert-base-uncased",
]

def eval_model(model_name: str):
    print(f"\n=== {model_name} ===")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Re-tokenize to 128 for THIS tokenizer (tokenizers differ!)
    def tok_fn(b):
        e = tok(b["text_norm"], truncation=True, padding=False, max_length=MAX_LEN)
        e["labels"] = b["label_norm"]; return e
    tok_ds_local = ds.map(tok_fn, batched=True, remove_columns=ds["train"].column_names)
    coll = DataCollatorWithPadding(tok)

    m = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(model.config.id2label)
    ).to(device)

    args = TrainingArguments(
        output_dir=f"runs/{model_name.replace('/','_')}-hx128",
        learning_rate=2e-5,
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        weight_decay=0.01,
        warmup_ratio=0.10,
        lr_scheduler_type="linear",
        eval_strategy="epoch",
        save_strategy="no",
        load_best_model_at_end=False,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        optim="adamw_torch",
    )

    tr = WeightedTrainer(
        model=m,
        args=args,
        train_dataset=tok_ds_local["train"],
        eval_dataset=tok_ds_local["validation"],
        tokenizer=tok,
        data_collator=coll,
        compute_metrics=compute_metrics,
        class_weights=(class_weights if 'class_weights' in globals() else None),
    )
    tr.train()
    metri = tr.evaluate(tok_ds_local["validation"])

    # clean up VRAM between runs
    del tr, m; torch.cuda.empty_cache(); gc.collect()
    return {"model": model_name, "macro_f1": metri["eval_macro_f1"], "accuracy": metri["eval_accuracy"]}

rows = []
for m in candidates:
    try:
        rows.append(eval_model(m))
    except RuntimeError as e:
        print(f"OOM or error on {m}: {e}")
df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
df



=== roberta-base ===


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (31 outputs left out)

Map:   0%|          | 0/1924 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,F1 Hatespeech,F1 Normal,F1 Offensive
1,0.761600,0.753983,0.690427,0.679320,0.680808,0.678841,0.759087,0.737562,0.541311
2,0.637400,0.726689,0.688345,0.681940,0.682684,0.689795,0.771363,0.710674,0.563782
3,0.481200,0.799919,0.666493,0.664831,0.678666,0.676591,0.766975,0.669718,0.557801
4,0.342100,0.863584,0.687305,0.681027,0.679705,0.684649,0.776699,0.717362,0.549020


,model,macro_f1,accuracy
0,roberta-base,0.689229,0.693548
3,bert-base-uncased,0.681027,0.687305
1,GroNLP/hateBERT,0.680686,0.686785
2,google/electra-base-discriminator,0.679771,0.688345


In [ ]:
# @title Before and After of roberta-base


from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd, numpy as np, torch, gc, time

def eval_model(model_name: str):
    print(f"\n=== {model_name} ===")
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    # Re-tokenize to 128 for THIS tokenizer (tokenizers differ!)
    def tok_fn(b):
        e = tok(b["text_norm"], truncation=True, padding=False, max_length=MAX_LEN)
        e["labels"] = b["label_norm"]; return e
    tok_ds_local = ds.map(tok_fn, batched=True, remove_columns=ds["train"].column_names)
    coll = DataCollatorWithPadding(tok)

    m = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(model.config.id2label)
    ).to(device)

    output_dir = f"runs/{model_name.replace('/','_')}-hx128" # Define output directory

    args = TrainingArguments(
        output_dir=output_dir, # Use the defined output directory
        learning_rate=2e-5,
        num_train_epochs=4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        weight_decay=0.01,
        warmup_ratio=0.10,
        lr_scheduler_type="linear",
        eval_strategy="epoch",
        #save_strategy="no", # Change to save the model
        save_strategy="epoch", # Save checkpoint every epoch
        load_best_model_at_end=False,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        optim="adamw_torch",
    )

    tr = WeightedTrainer(
        model=m,
        args=args,
        train_dataset=tok_ds_local["train"],
        eval_dataset=tok_ds_local["validation"],
        tokenizer=tok,
        data_collator=coll,
        compute_metrics=compute_metrics,
        class_weights=(class_weights if 'class_weights' in globals() else None),
    )

    # Evaluate BEFORE training
    print("Evaluating BEFORE training...")
    metrics_before = tr.evaluate(tok_ds_local["validation"])
    print("Metrics BEFORE training:", metrics_before)


    tr.train()

    # Evaluate AFTER training
    print("Evaluating AFTER training...")
    metrics_after = tr.evaluate(tok_ds_local["validation"])
    print("Metrics AFTER training:", metrics_after)

    # Save the fine-tuned model
    tr.save_model(output_dir)
    print(f"Model saved to {output_dir}")


    # clean up VRAM between runs
    del tr, m; torch.cuda.empty_cache(); gc.collect()
    return {
        "model": model_name,
        "macro_f1_before": metrics_before["eval_macro_f1"],
        "accuracy_before": metrics_before["eval_accuracy"],
        "macro_f1_after": metrics_after["eval_macro_f1"],
        "accuracy_after": metrics_after["eval_accuracy"],
        "saved_path": output_dir # Add saved path to results
    }


# rows = []
# for m in candidates:
#     try:
#         rows.append(eval_model(m))
#     except RuntimeError as e:
#         print(f"OOM or error on {m}: {e}")
# df = pd.DataFrame(rows).sort_values("macro_f1_after", ascending=False)
# df

# Direct call for roberta-base
try:
    result = eval_model("roberta-base")
    df = pd.DataFrame([result]) # Create a DataFrame with a single row
    display(df)
except RuntimeError as e:
    print(f"OOM or error on roberta-base: {e}")


=== roberta-base ===


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (8 outputs left out)

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,Macro F1,Macro Precision,Macro Recall,F1 Hatespeech,F1 Normal,F1 Offensive
1,0.768200,0.758648,0.003200,0.685224,0.654066,0.679483,0.660403,0.777778,0.737908,0.446512
2,0.669500,0.744040,0.003200,0.677940,0.670502,0.670787,0.678085,0.769468,0.704403,0.537634
3,0.543900,0.758992,0.003200,0.670135,0.667185,0.675022,0.676315,0.766123,0.686347,0.549085
4,0.446600,0.815742,0.003200,0.695109,0.689379,0.689323,0.694535,0.782470,0.726018,0.559649


Evaluating AFTER training...


Metrics AFTER training: {'eval_loss': 0.815742015838623, 'eval_model_preparation_time': 0.0032, 'eval_accuracy': 0.6951092611862643, 'eval_macro_f1': 0.6893790619127259, 'eval_macro_precision': 0.6893227813174173, 'eval_macro_recall': 0.6945351871748486, 'eval_f1_hatespeech': 0.7824701195219124, 'eval_f1_normal': 0.7260179434092477, 'eval_f1_offensive': 0.5596491228070175, 'eval_runtime': 1.936, 'eval_samples_per_second': 992.745, 'eval_steps_per_second': 16.012, 'epoch': 4.0}


,model,macro_f1_before,accuracy_before,macro_f1_after,accuracy_after
0,roberta-base,0.268738,0.364204,0.689379,0.695109


In [ ]:
!pip -q install peft

In [ ]:
# @title Training with PEFT

from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification
import torch, gc
import pandas as pd

# Define LoRA configuration
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["query", "value"], # Modules to apply LoRA to
    lora_dropout=0.1, # Dropout probability for LoRA layers
    bias="none", # Bias type for LoRA
    task_type="SEQ_CLS", # Task type for sequence classification
)

# Load the base RoBERTa model again for LoRA
lora_model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base", num_labels=len(ID2NAME)
).to(device)

# Apply LoRA to the model
lora_model = get_peft_model(lora_model, lora_config)

# Print trainable parameters
lora_model.print_trainable_parameters()

lora_output_dir = "runs/roberta-base-hx128-lora" # Define output directory

# Configure TrainingArguments for LoRA (can be similar to full fine-tuning)
lora_training_args = TrainingArguments(
    output_dir=lora_output_dir,
    learning_rate=2e-4,
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    warmup_ratio=0.10,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    #save_strategy="no", # Change to save the model
    save_strategy="epoch", # Save checkpoint every epoch
    load_best_model_at_end=False,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    report_to="none",
    optim="adamw_torch",
)

# Create a Trainer for LoRA
lora_trainer = WeightedTrainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=tok_ds_128["train"], # Use the already tokenized dataset
    eval_dataset=tok_ds_128["validation"],
    tokenizer=tokenizer, # Use the RoBERTa tokenizer
    data_collator=collator, # Use the DataCollatorWithPadding
    compute_metrics=compute_metrics,
    class_weights=(class_weights if 'class_weights' in globals() else None),
)

# Evaluate BEFORE training (LoRA weights are initialized to zero, so should be similar to base model)
print("Evaluating LoRA model BEFORE training...")
lora_metrics_before = lora_trainer.evaluate(tok_ds_128["validation"])
print("LoRA Metrics BEFORE training:", lora_metrics_before)

# Train the LoRA model
print("Training LoRA model...")
lora_trainer.train()

# Evaluate AFTER training
print("Evaluating LoRA model AFTER training...")
lora_metrics_after = lora_trainer.evaluate(tok_ds_128["validation"])
print("LoRA Metrics AFTER training:", lora_metrics_after)

# Save the LoRA model
lora_trainer.save_model(lora_output_dir)
print(f"LoRA model saved to {lora_output_dir}")


# Display results
lora_result = {
    "model": "roberta-base (LoRA)",
    "macro_f1_before": lora_metrics_before["eval_macro_f1"],
    "accuracy_before": lora_metrics_before["eval_accuracy"],
    "macro_f1_after": lora_metrics_after["eval_macro_f1"],
    "accuracy_after": lora_metrics_after["eval_accuracy"],
    "saved_path": lora_output_dir # Add saved path to results
}
lora_df = pd.DataFrame([lora_result])
display(lora_df)

# Clean up VRAM
del lora_trainer, lora_model; torch.cuda.empty_cache(); gc.collect()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,811 || all params: 125,535,750 || trainable%: 0.7072
Evaluating LoRA model BEFORE training...


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


LoRA Metrics BEFORE training: {'eval_loss': 1.098657488822937, 'eval_model_preparation_time': 0.0087, 'eval_accuracy': 0.2939646201873049, 'eval_macro_f1': 0.19644941139987118, 'eval_macro_precision': 0.2282667849748595, 'eval_macro_recall': 0.3293378942088719, 'eval_f1_hatespeech': 0.0, 'eval_f1_normal': 0.1617195496417605, 'eval_f1_offensive': 0.4276286845578531, 'eval_runtime': 2.4918, 'eval_samples_per_second': 771.328, 'eval_steps_per_second': 12.441}
Training LoRA model...


Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy,Macro F1,Macro Precision,Macro Recall,F1 Hatespeech,F1 Normal,F1 Offensive
1,0.960900,0.923822,0.008700,0.573881,0.543293,0.570720,0.545927,0.655647,0.637867,0.336364
2,0.867200,0.865102,0.008700,0.616545,0.596555,0.620797,0.593969,0.696203,0.662259,0.431203
3,0.818000,0.843999,0.008700,0.603018,0.599849,0.598949,0.603528,0.691619,0.613605,0.494323
4,0.812000,0.838357,0.008700,0.630593,0.629423,0.642735,0.623873,0.705441,0.657143,0.525685
5,0.822300,0.824826,0.008700,0.638918,0.628658,0.638749,0.624516,0.707143,0.676887,0.501946
6,0.780200,0.830787,0.008700,0.642560,0.635300,0.641286,0.631813,0.705778,0.679268,0.520853
7,0.739100,0.836268,0.008700,0.637877,0.632314,0.638528,0.628562,0.709042,0.671596,0.516304
8,0.754400,0.831093,0.008700,0.640479,0.630898,0.632407,0.630493,0.715008,0.675776,0.501908


Evaluating LoRA model AFTER training...


LoRA Metrics AFTER training: {'eval_loss': 0.8310931921005249, 'eval_model_preparation_time': 0.0087, 'eval_accuracy': 0.6404786680541104, 'eval_macro_f1': 0.6308977420550989, 'eval_macro_precision': 0.6324069118946346, 'eval_macro_recall': 0.6304927775761554, 'eval_f1_hatespeech': 0.715008431703204, 'eval_f1_normal': 0.675776397515528, 'eval_f1_offensive': 0.5019083969465649, 'eval_runtime': 2.2386, 'eval_samples_per_second': 858.58, 'eval_steps_per_second': 13.848, 'epoch': 8.0}


,model,macro_f1_before,accuracy_before,macro_f1_after,accuracy_after
0,roberta-base (LoRA),0.196449,0.293965,0.630898,0.640479


271

In [ ]:
# @title Comparing roberta full fine-tune & LoRA


# Combine the results into a list of dictionaries
comparison_data = [result, lora_result]

# Create a DataFrame from the combined data
comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
display(comparison_df)

,model,macro_f1_before,accuracy_before,macro_f1_after,accuracy_after
0,roberta-base,0.268738,0.364204,0.689379,0.695109
1,roberta-base (LoRA),0.196449,0.293965,0.608902,0.616545


In [ ]:
# @title
import torch
from torch.utils.data import DataLoader
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc
import os

# Explicitly delete previous models and trainers to free up memory
if 'lora_model' in globals() and lora_model is not None:
    del lora_model
if 'lora_trainer' in globals() and lora_trainer is not None:
    del lora_trainer
if 'm' in globals() and m is not None:
    del m
if 'tr' in globals() and tr is not None:
    del tr
if 'trainer_hate' in globals() and trainer_hate is not None:
    del trainer_hate
if 'm_hate' in globals() and m_hate is not None:
    del m_hate

torch.cuda.empty_cache()
gc.collect()
print("GPU memory cleared before manual evaluation.")

MODEL_NAME_HATE = "Hate-speech-CNERG/bert-base-uncased-hatexplain"
print(f"\n=== Manually Evaluating {MODEL_NAME_HATE} (no training) ===")

# Load tokenizer and model
tok_hate = AutoTokenizer.from_pretrained(MODEL_NAME_HATE, use_fast=True)
m_hate = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_HATE, num_labels=len(ID2NAME)
).to(device)
m_hate.eval() # Set model to evaluation mode

# Prepare the dataset and dataloader
# Re-tokenize for this specific tokenizer if needed (HateXplain tokenizer should be compatible with bert-base-uncased)
def tok_fn_hate_eval(b):
    # Explicitly pad to max_length during tokenization
    e = tok_hate(b["text_norm"], truncation=True, padding='max_length', max_length=MAX_LEN, return_tensors="pt")
    e["labels"] = torch.tensor(b["label_norm"], dtype=torch.long)
    return e

# Map the tokenization function without removing columns initially
tok_ds_local_hate_eval = ds["validation"].map(tok_fn_hate_eval, batched=True)

# Remove original columns after tokenization, keep only processed tensors and labels
columns_to_keep = ['input_ids', 'attention_mask', 'labels']
tok_ds_local_hate_eval = tok_ds_local_hate_eval.remove_columns([col for col in tok_ds_local_hate_eval.column_names if col not in columns_to_keep])

# Set dataset format for PyTorch
tok_ds_local_hate_eval.set_format("torch", columns=columns_to_keep)


eval_dataloader = DataLoader(
    tok_ds_local_hate_eval,
    batch_size=16, # Use a small batch size for evaluation
    shuffle=False,
    collate_fn=DataCollatorWithPadding(tok_hate, return_tensors="pt"), # Use collator for padding within batches
)

all_preds = []
all_labels = []

# Manual evaluation loop
with torch.no_grad():
    for batch in eval_dataloader:
        # Move batch to device
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")

        # Forward pass
        outputs = m_hate(**batch)
        logits = outputs.logits

        # Move predictions and labels to CPU and store
        all_preds.append(logits.argmax(dim=-1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

# Concatenate all predictions and labels
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

# Compute metrics manually
acc = evaluate.load("accuracy")
f1  = evaluate.load("f1")
prec= evaluate.load("precision")
rec = evaluate.load("recall")

metrics_hate = {
    "eval_accuracy": acc.compute(predictions=all_preds, references=all_labels)["accuracy"],
    "eval_macro_f1": f1.compute(predictions=all_preds, references=all_labels, average="macro")["f1"],
    "eval_macro_precision": prec.compute(predictions=all_preds, references=all_labels, average="macro")["precision"],
    "eval_macro_recall": rec.compute(predictions=all_preds, references=all_labels, average="macro")["recall"],
}

# per-class F1
for i, name in enumerate(ID2NAME.values()):
    metrics_hate[f"eval_f1_{name}"] = f1.compute(
        predictions=(all_preds==i).astype(int),
        references=(all_labels==i).astype(int),
        average="binary"
    )["f1"]


print("Metrics for", MODEL_NAME_HATE, ":", metrics_hate)

# Save the evaluated HateXplain model
hate_model_output_dir = f"runs/{MODEL_NAME_HATE.replace('/','_')}_evaluated"
os.makedirs(hate_model_output_dir, exist_ok=True) # Create directory if it doesn't exist
m_hate.save_pretrained(hate_model_output_dir)
tok_hate.save_pretrained(hate_model_output_dir) # Also save the tokenizer
print(f"Evaluated HateXplain model saved to {hate_model_output_dir}")


# Clean up VRAM
del m_hate, eval_dataloader, all_preds, all_labels; torch.cuda.empty_cache(); gc.collect()

# Optionally add to comparison table
hate_result = {
    "model": MODEL_NAME_HATE + " (Evaluated)",
    "macro_f1_before": "na",
    "accuracy_before": "na",
    "macro_f1_after": metrics_hate["eval_macro_f1"],
    "accuracy_after": metrics_hate["eval_accuracy"],
    "saved_path": hate_model_output_dir # Add saved path to results
}
if 'comparison_data' in globals():
    comparison_data.append(hate_result)
else:
    comparison_data = [hate_result]
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

GPU memory cleared before manual evaluation.

=== Manually Evaluating Hate-speech-CNERG/bert-base-uncased-hatexplain (no training) ===


Map:   0%|          | 0/1922 [00:00<?, ? examples/s]

Metrics for Hate-speech-CNERG/bert-base-uncased-hatexplain : {'eval_accuracy': 0.6836628511966701, 'eval_macro_f1': 0.679997028621723, 'eval_macro_precision': 0.6814507446786869, 'eval_macro_recall': 0.6796502854884013, 'eval_f1_hatespeech': 0.7583834909716252, 'eval_f1_normal': 0.7131681877444589, 'eval_f1_offensive': 0.5684394071490846}


,model,macro_f1_before,accuracy_before,macro_f1_after,accuracy_after
0,roberta-base,0.268738,0.364204,0.689379,0.695109
1,roberta-base (LoRA),0.196449,0.293965,0.608902,0.616545
2,Hate-speech-CNERG/bert-base-uncased-hatexplain...,0.679997,0.683663,0.679997,0.683663


In [ ]:
# @title Comparing with the paper's model

hate_result = {
    "model": MODEL_NAME_HATE + " (Evaluated)",
    "macro_f1_before": "na",
    "accuracy_before": "na",
    "macro_f1_after": metrics_hate["eval_macro_f1"],
    "accuracy_after": metrics_hate["eval_accuracy"],
}

# Start with a fresh list for comparison data or find and replace the existing one
if 'comparison_data' in globals():
    # Find if an entry for this model already exists and replace it
    found_index = -1
    for i, entry in enumerate(comparison_data):
        if entry.get("model") == hate_result["model"]:
            found_index = i
            break
    if found_index != -1:
        comparison_data[found_index] = hate_result
    else:
        comparison_data.append(hate_result)
else:
    comparison_data = [result, lora_result, hate_result] # Initialize with all results if not already present

# Ensure only one entry for each model type exists (handles initial runs and re-runs)
unique_comparison_data = {}
if 'result' in globals():
    unique_comparison_data[result["model"]] = result
if 'lora_result' in globals():
    unique_comparison_data[lora_result["model"]] = lora_result
if 'hate_result' in globals():
     # Use the key that matches the 'model' value in hate_result
    unique_comparison_data[hate_result["model"]] = hate_result


comparison_data = list(unique_comparison_data.values())


comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

,model,macro_f1_before,accuracy_before,macro_f1_after,accuracy_after
0,roberta-base,0.268738,0.364204,0.689379,0.695109
1,roberta-base (LoRA),0.196449,0.293965,0.630898,0.640479
2,Hate-speech-CNERG/bert-base-uncased-hatexplain...,na,na,0.679997,0.683663


### Model Inference

Use the cell below to run inference on the trained models. You can input your own text and see how each model classifies it.

In [ ]:
# @title Run Inference

input_text = "This is a test sentence." # @param {type:"string"}

# Paths to the saved models
FULL_TUNED_MODEL_PATH = "runs/roberta-base-hx128" # Must match output_dir in full fine-tuning cell
LORA_MODEL_PATH = "runs/roberta-base-hx128-lora" # Must match lora_output_dir in LoRA cell
HATE_MODEL_PATH = "runs/Hate-speech-CNERG_bert-base-uncased-hatexplain_evaluated" # Must match hate_model_output_dir in HateXplain eval cell


# Function to perform inference on a given model and tokenizer
def run_inference(model, tokenizer, text, device):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Perform inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Get the predicted class
    predicted_class_id = logits.argmax(dim=-1).item()
    predicted_label = ID2NAME[predicted_class_class_id]

    return predicted_label

print(f"Input Text: {input_text}")
print("-" * 30)

# --- Inference with Full Fine-tuned RoBERTa ---
print(f"Loading Full Fine-tuned model from {FULL_TUNED_MODEL_PATH} for inference...")
try:
    # Load the tokenizer and model from the saved path
    full_tuned_tokenizer = AutoTokenizer.from_pretrained(FULL_TUNED_MODEL_PATH)
    full_tuned_model = AutoModelForSequenceClassification.from_pretrained(
        FULL_TUNED_MODEL_PATH, num_labels=len(ID2NAME)
    ).to(device)

    print("Inference with Full Fine-tuned RoBERTa:")
    full_tuned_model.eval() # Set model to evaluation mode
    predicted_label_full_tune = run_inference(full_tuned_model, full_tuned_tokenizer, input_text, device)
    print(f"Predicted Label: {predicted_label_full_tune}")

    del full_tuned_model; torch.cuda.empty_cache(); gc.collect()

except Exception as e:
    print(f"Error loading or running inference with Full Fine-tuned RoBERTa: {e}")
    print(f"Please ensure the full fine-tuning cell was run successfully and the model was saved to {FULL_TUNED_MODEL_PATH}")


print("-" * 30)

# --- Inference with LoRA RoBERTa ---
print(f"Loading LoRA model from {LORA_MODEL_PATH} for inference...")
try:
    from peft import PeftModel, PeftConfig
    # Load the base model first
    lora_base_model = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base", num_labels=len(ID2NAME)
    ).to(device)

    # Load the LoRA adapters onto the base model
    lora_model_inf = PeftModel.from_pretrained(lora_base_model, LORA_MODEL_PATH)

    print("Inference with LoRA RoBERTa:")
    lora_model_inf.eval() # Set model to evaluation mode
    # Use the original RoBERTa tokenizer for the LoRA model
    predicted_label_lora = run_inference(lora_model_inf, tokenizer, input_text, device)
    print(f"Predicted Label: {predicted_label_lora}")

    del lora_base_model, lora_model_inf; torch.cuda.empty_cache(); gc.collect()

except Exception as e:
    print(f"Error loading or running inference with LoRA RoBERTa: {e}")
    print(f"Please ensure the LoRA fine-tuning cell was run successfully and the model was saved to {LORA_MODEL_PATH}")


print("-" * 30)

# --- Inference with Pre-trained HateXplain Model ---
print(f"Loading Pre-trained HateXplain model from {HATE_MODEL_PATH} for inference...")
try:
    # Load the tokenizer and model from the saved path
    hate_tokenizer_inf = AutoTokenizer.from_pretrained(HATE_MODEL_PATH)
    hate_model_inf = AutoModelForSequenceClassification.from_pretrained(
        HATE_MODEL_PATH, num_labels=len(ID2NAME)
    ).to(device)

    print("Inference with Pre-trained HateXplain Model:")
    hate_model_inf.eval() # Set model to evaluation mode
    predicted_label_hate = run_inference(hate_model_inf, hate_tokenizer_inf, input_text, device)
    print(f"Predicted Label: {predicted_label_hate}")

    del hate_model_inf; torch.cuda.empty_cache(); gc.collect()

except Exception as e:
    print(f"Error loading or running inference with Pre-trained HateXplain model: {e}")
    print(f"Please ensure the HateXplain evaluation cell was run successfully and the model was saved to {HATE_MODEL_PATH}")


print("-" * 30)
print("Inference complete. GPU memory cleared after inference.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Input Text: why cant these people just go back to where they came from, fucking assholes
------------------------------
Loading roberta-base for inference...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Inference with Full Fine-tuned RoBERTa:
Predicted Label: offensive
------------------------------
Loading roberta-base with LoRA for inference...
Inference with LoRA RoBERTa:
Predicted Label: offensive
------------------------------
Loading Hate-speech-CNERG/bert-base-uncased-hatexplain for inference...
Inference with Pre-trained HateXplain Model:
Predicted Label: offensive
------------------------------
Inference complete. GPU memory cleared after inference.
